# Phase 1 · Dataset selection and justification

**Project 1 · The Five Classes of Learners · Machine Learning Foundations (UA 41010)**

**Team project** (2 students), University of Aveiro

**What this notebook does:** it collects, in an orderly way, what we decided *before* touching any data: which problem we solve, why this dataset, what we expect from each learner and what the limits are. It is the basis of section 3 of the README.

## 1. Project objective

Pedro Domingos groups machine learning into **five classes of learners**, each with its own idea of what it means to *learn*. We have to choose **one learner from each class**, train the five on the **same** dataset and compare them with a methodology that we define and defend.

The main question is **not "which model won?"** but **"why?"**: what each class assumes about the data and what the plots and metrics say about those assumptions.

Assignment rules that condition everything else:
- one learner per class (five in total); different configurations of the same learner count as one;
- tuning a hyperparameter with an evolutionary algorithm does **not** make a learner evolutionary: evolution must be the mechanism by which the model learns;
- every preprocessing statistic is learned **only from the training part** of each fold;
- fixed seeds; the test set is used **only once**, at the end;
- it must run in a few minutes on a laptop.

## 2. The five classes of learners

| Class | Learns by | Learner we use | Idea in one sentence |
|---|---|---|---|
| **Symbolists** | rule induction | Decision tree | Asks yes/no questions about the features |
| **Connectionists** | adjusting weights by backpropagation | MLP (neural network) | Weights corrected with the error |
| **Evolutionaries** | evolving a population of candidates | Linear classifier evolved with differential evolution (pyBlindOpt) | Mutate and select the best weights, optimizing the metric directly |
| **Bayesians** | updating probabilities with Bayes' rule | Gaussian Naive Bayes | Probability of each class given the evidence |
| **Analogizers** | comparing a new case with similar ones | SVM with a kernel | A new tumour resembles the ones we already know |

*(The details of each model and its hyperparameter space are explained in Phases 4 and 5.)*

## 3. Classification or regression?

| | Regression | Classification |
|---|---|---|
| **What it predicts** | a continuous real number | a category (yes/no or others) |
| **How it is measured** | RMSE, MAE, R² | confusion matrix, recall, F1, MCC |
| **When it is interesting** | noise, limits, outliers | decisions with **different costs** depending on the error |

**Choice: classification.** In medicine the two errors do not cost the same, and that lets us discuss something very important: **changing the decision threshold** to reduce the real cost.

## 4. Chosen dataset: `breast_cancer`

Classification options offered by the course:

| Dataset | Size | What makes it interesting |
|---|---|---|
| **breast_cancer** ✅ | 569 × 30 | clean and easy: ceiling effect, highly correlated features, **cost of missing a tumour** |
| german_credit | 1000 × 20 | numeric and categorical; errors with a 5:1 cost |
| heart_disease | 303 × 13 | very small, missing values: variance is the problem |
| spambase | 4601 × 57 | very skewed, many zeros, duplicates |
| digits | 1797 × 64 | multiclass, images as vectors |

### Why we chose it
1. **Motivation:** we want to apply machine learning to a real problem of critical impact (health). Cancer is a reality close to all of us; we are interested in how AI can be a **support for diagnosis**, never a substitute for the professional.
2. **Technical reason:** the **false negative** (saying "benign" to a malignant tumour) costs far more than the **false positive** (a false alarm that leads to more tests). That is why accuracy is not enough: we have to look at **recall** and the **threshold**.
3. **Analytical challenge:** since it is clean, the challenge is not cleaning data but seeing how each family of models handles the **redundancy** of the information up to the **ceiling effect**.

### Task, objective and value
- **Task:** binary classification: is the tumour benign or malignant?
- **Input:** 30 numeric features of the cell nuclei (10 measurements × mean, standard error and worst value).
- **Objective:** compare five models from different families, their results and **which errors each one makes**.
- **Value of a good model:** a support tool (triage, a "second opinion" that never gets tired) that helps decide which cases need special attention, **without replacing** the doctor.

### First look at the data (shape and header only, no exploration)

In [1]:
import polars as pl  # polars: reads and manipulates the data table
from pathlib import Path  # Path: handles file paths

path = Path("datasets/breast_cancer.csv.zst")  # the data file is in the datasets folder
data = pl.read_csv(path)  # read the compressed file
print("Rows (tumours):", data.height)  # number of rows
print("Columns:", data.width)  # 30 features + 1 target column
print("Target column:", "target")  # the diagnosis
print("Note: in the file target=1 is BENIGN; in Phase 2 we recode it as malignant=1")  # important warning
data.head(3)  # first 3 rows

Rows (tumours): 569
Columns: 31
Target column: target
Note: in the file target=1 is BENIGN; in Phase 2 we recode it as malignant=1


mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,perimeter_error,area_error,smoothness_error,compactness_error,concavity_error,concave_points_error,symmetry_error,fractal_dimension_error,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,target
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
17.99,10.38,122.8,1001.0,0.1184,0.2776,0.3001,0.1471,0.2419,0.07871,1.095,0.9053,8.589,153.4,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189,0
20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.0186,0.0134,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.186,0.275,0.08902,0
19.69,21.25,130.0,1203.0,0.1096,0.1599,0.1974,0.1279,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.00615,0.04006,0.03832,0.02058,0.0225,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.243,0.3613,0.08758,0


## 5. Starting hypotheses (per class)

**Central idea:** the success of the learners will depend not on drawing complex boundaries, but on **how each paradigm handles redundant evidence** and on **whether it allows the threshold to be adjusted to the medical cost**.

| Class | Hypothesis (what we predict) |
|---|---|
| **Bayesian** (Naive Bayes) | The features are highly correlated, but Naive Bayes treats them as independent: it **counts the same evidence several times**. Its probabilities will be **overconfident** (close to 0 or 1) and it will be the one that adjusts the threshold worst. |
| **Connectionist** (MLP) | It will **dominate**: regularization reduces the weight of the redundant features and its sigmoid output gives reasonably calibrated probabilities, which allows the threshold to be moved precisely. |
| **Symbolic** (tree) | **Very interpretable** (an oncologist can read it) and it ignores redundancy by choosing one feature per node. But its hard cuts give coarse probabilities, so it will calibrate risk worse. |
| **Analogizer** (SVM / k-NN) | Redundant features **distort distances** (size counts three times: radius, perimeter, area). Without preprocessing it will need a lot of tuning and it gives no direct probability. |
| **Evolutionary** (differential evolution) | It can optimize the MCC or an asymmetric cost directly, but on a clean, almost convex problem **blind search will be very inefficient** compared with a gradient method. |

**Expected conclusion:** in health, with redundancy and errors that cost lives, the winner will not be the model with the most complex boundary, but the one that allows **better risk calibration and regularization of the evidence**.

## 6. Limits of the dataset

| # | Limit | Consequence for the project |
|---|---|---|
| 1 | **Small size** (569 rows, 30 features): high variance; a single test split gives unstable results | **repeated** cross-validation and always mean **± standard deviation** |
| 2 | **Ceiling effect**: almost any reasonable model scores > 95% | statistical tests (corrected t, McNemar) to know whether a difference is real or noise |
| 3 | **Collinearity**: radius, perimeter and area measure almost the same thing | regularization; we expect poor calibration in Naive Bayes |
| 4 | **Asymmetric errors**: the false negative costs much more | MCC, recall and F1 (not only accuracy); **threshold adjustment** |
| 5 | It is **a single source** (one hospital, one type of measurement) | we cannot claim that it works in other hospitals or that it diagnoses real patients |

## 7. Value of a good model in the real world

1. **Safety net (triage):** a "second opinion" that does not get tired; it applies the same rigour to the first and the last case of the shift.
2. **Minimising fatal errors:** a well-calibrated model can be configured conservatively so that **no malignant tumour is sent home** without further review.
3. **Controlling false alarms:** by confidently classifying the clearly benign tumours, it avoids unnecessary surgeries and leaves the specialist's time for the cases at risk.

## 8. Caveats to keep in mind (methodological honesty)

- **The 5:1 cost is our assumption.** The assignment gives no cost matrix for `breast_cancer`; we take the 5:1 from `german_credit` and declare it. That is why the main metric is the **MCC**, and the cost is used as an additional analysis.
- **The hypotheses are there to be tested, not defended.** If the results contradict them, we say so (Phase 6).
- **Course stack:** the assignment mentions JAX/Keras. For the connectionist learner we use scikit-learn's `MLPClassifier`, which the assignment's table allows. It remains declared as a limit.
- **Caveat on the analogizer:** the distorted-distances hypothesis applies mostly to k-NN. With an RBF-kernel SVM distances are also measured, but the margin and regularization soften the effect.

**How it ended (detailed in Phase 6):** there was no clear winner; the tree was the only one significantly worse, and the rest tied because of the ceiling effect. The hypothesis that the connectionist learner would *dominate* was **not** confirmed.

## Summary of Phase 1

| Question | Answer |
|---|---|
| Task | binary classification |
| Dataset | `breast_cancer` (569 × 30) |
| Objective | benign or malignant tumour? |
| Learners | tree · MLP · evolutionary · Naive Bayes · SVM |
| Central idea | what matters is **calibration and the threshold with asymmetric costs**, not the most complex boundary |
| Limits | little data, ceiling effect, collinearity, asymmetric errors |

**Next:** Phase 2 · Data visualization.